# Кросс-валидация

#### 1. Загрузите датасет ирисы Фишера из библиотеки sklearn.datasets.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target
print("Shapes:", X.shape, y.shape)
print("Feature names:", iris.feature_names)
print("Target names:", iris.target_names)

iris_df = pd.DataFrame(X, columns=iris.feature_names)
iris_df["target"] = y
print("\nПервые 5 строк:")
display(iris_df.head())

Shapes: (150, 4) (150,)
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target names: ['setosa' 'versicolor' 'virginica']

Первые 5 строк:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


#### 2. Сделайте hold-out разбиение данных. Для этого разделите данные на обучающую и валидационную выборки и выведите на экран соответствующие индексы разбиения.

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split

indices = np.arange(len(X))
train_indices, test_indices = train_test_split(
    indices, test_size=0.30, shuffle=False
)

print("Hold-out split (без перемешивания)")
print("Train indices:", train_indices)
print("Test indices:", test_indices)

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

Hold-out split (без перемешивания)
Train indices: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104]
Test indices: [105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122
 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140
 141 142 143 144 145 146 147 148 149]


#### 3. Теперь сделайте разбиение перемешанных данных, зафиксировав воспроизводимость выбора данных после перемешивания, указав значение параметра random_state=42 и выведите на экран соответствующие индексы разбиения.

In [13]:
shuffled_indices = np.arange(len(X))
train_idx, test_idx = train_test_split(
    shuffled_indices, test_size=0.30, shuffle=True, random_state=42
)

print("Hold-out split (с перемешиванием, random_state=42)")
print("Train indices:", train_idx)
print("Test indices:", test_idx)

X_train_s, X_test_s = X[train_idx], X[test_idx]
y_train_s, y_test_s = y[train_idx], y[test_idx]

Hold-out split (с перемешиванием, random_state=42)
Train indices: [ 81 133 137  75 109  96 105  66   0 122  67  28  40  44  60 123  24  25
  23  94  39  95 117  47  97 113  33 138 101  62  84 148  53   5  93 111
  49  35  80  77  34 114   7  43  70  98 120  83 134 135  89   8  13 119
 125   3  17  38  72 136   6 112 100   2  63  54 126  50 115  46 139  61
 147  79  59  91  41  58  90  48  88 107 124  21  57 144 129  37 140   1
  52 130 103  99 116  87  74 121 149  20  71 106  14  92 102]
Test indices: [ 73  18 118  78  76  31  64 141  68  82 110  12  36   9  19  56 104  69
  55 132  29 127  26 128 131 145 108 143  45  30  22  15  65  11  42 146
  51  27   4  32 142  85  86  16  10]


#### 4. Обучите модель логистической регрессии на обучающих данных. Выведите значения коэффициентов модели, полученных в результате обучения. Сделайте предсказание на тестовом наборе признаков. Выведите значение метрик accuracy и f1-score.

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

model = LogisticRegression(solver='lbfgs', max_iter=1000)
model.fit(X_train_s, y_train_s)

print("Коэффициенты модели (hold-out 70/30, shuffled):")
print(model.coef_)

y_pred = model.predict(X_test_s)
acc_holdout_70 = accuracy_score(y_test_s, y_pred)
f1_holdout_70 = f1_score(y_test_s, y_pred, average='macro')

print("Accuracy:", acc_holdout_70)
print("F1-score (macro):", f1_holdout_70)

Коэффициенты модели (hold-out 70/30, shuffled):
[[-0.40538525  0.86892324 -2.27787476 -0.95680108]
 [ 0.46642678 -0.37487862 -0.18745264 -0.72127186]
 [-0.06104153 -0.49404462  2.4653274   1.67807293]]
Accuracy: 1.0
F1-score (macro): 1.0


#### 5. Разделите данные на обучающую и валидационную выборки по новому в соотношении 75-25. Обучите модель на этих данных, выведите значения получившихся коэффициентов модели. Выведите значения метрик и сравните их со значениями из предыдущего пункта. Сделайте вывод о том, влияет ли способ разбиения на результат.

In [16]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)

model2 = LogisticRegression(solver='lbfgs', max_iter=1000)
model2.fit(X_train2, y_train2)

print("Коэффициенты модели (75/25):")
print(model2.coef_)

y_pred2 = model2.predict(X_test2)
acc_holdout_75 = accuracy_score(y_test2, y_pred2)
f1_holdout_75 = f1_score(y_test2, y_pred2, average='macro')

print("Accuracy (75/25):", acc_holdout_75)
print("F1-score (macro, 75/25):", f1_holdout_75)

print("\nСравнение с предыдущим hold-out (70/30):")
print("ΔAccuracy:", acc_holdout_75 - acc_holdout_70)
print("ΔF1:", f1_holdout_75 - f1_holdout_70)

if (acc_holdout_75 != acc_holdout_70) or (f1_holdout_75 != f1_holdout_70):
    print("Вывод: способ разбиения влияет на итоговые метрики.")
else:
    print("Вывод: в этом запуске метрики совпали, но в целом способ разбиения может влиять на результат.")

Коэффициенты модели (75/25):
[[-0.39086522  0.92121445 -2.33169485 -0.9799742 ]
 [ 0.49862406 -0.30952765 -0.21642636 -0.73163851]
 [-0.10775883 -0.6116868   2.54812121  1.7116127 ]]
Accuracy (75/25): 1.0
F1-score (macro, 75/25): 1.0

Сравнение с предыдущим hold-out (70/30):
ΔAccuracy: 0.0
ΔF1: 0.0
Вывод: в этом запуске метрики совпали, но в целом способ разбиения может влиять на результат.


#### 6. Теперь сделайте k-блочную перекрёстную проверку модели (кросс-валидацию). Сравните полученные метрики с метриками, которые были при hold-out разбиении.

In [17]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=3, shuffle=True, random_state=15)
manual_kfold_acc = []
manual_kfold_f1 = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    print(f"Fold {fold}: train={train_idx}, test={test_idx}")
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    fold_model = LogisticRegression(solver='lbfgs', max_iter=1000)
    fold_model.fit(X_train, y_train)
    y_pred = fold_model.predict(X_test)

    manual_kfold_acc.append(accuracy_score(y_test, y_pred))
    manual_kfold_f1.append(f1_score(y_test, y_pred, average='macro'))

print("\nKFold (manual) Accuracy:", manual_kfold_acc)
print("KFold (manual) F1_macro:", manual_kfold_f1)
print("KFold mean Accuracy:", np.mean(manual_kfold_acc))
print("KFold mean F1_macro:", np.mean(manual_kfold_f1))
print("Сравнение с hold-out 70/30 (mean CV - hold-out):")
print("ΔAccuracy:", np.mean(manual_kfold_acc) - acc_holdout_70)
print("ΔF1:", np.mean(manual_kfold_f1) - f1_holdout_70)

Fold 1: train=[  1   2   3   4   7  10  14  15  16  17  18  19  22  23  24  26  28  29
  32  33  34  35  37  38  39  40  41  42  43  44  45  46  49  50  51  52
  53  54  56  60  62  63  64  65  66  68  69  70  73  75  76  77  79  80
  81  82  83  85  87  88  91  92  93  94  96  99 101 102 104 105 106 107
 108 110 111 113 114 117 118 119 120 121 123 125 128 131 132 133 134 135
 136 137 139 140 141 142 144 145 146 147], test=[  0   5   6   8   9  11  12  13  20  21  25  27  30  31  36  47  48  55
  57  58  59  61  67  71  72  74  78  84  86  89  90  95  97  98 100 103
 109 112 115 116 122 124 126 127 129 130 138 143 148 149]
Fold 2: train=[  0   1   4   5   6   7   8   9  10  11  12  13  15  17  19  20  21  22
  23  24  25  26  27  28  30  31  34  36  37  39  40  41  42  44  47  48
  50  53  55  56  57  58  59  60  61  62  63  65  66  67  70  71  72  74
  75  78  79  84  85  86  89  90  95  96  97  98  99 100 101 102 103 104
 105 107 109 112 114 115 116 118 119 121 122 124 125 126 127 12

#### 7. Теперь сделайте ту же самую перекрёстную проверку модели, используя библиотечную функцию cross_val_score. Убедитесь, что получится тот же результат.

In [18]:
from sklearn.model_selection import cross_val_score

cv_model = LogisticRegression(solver='lbfgs', max_iter=1000)
kfold_acc_cv = cross_val_score(cv_model, X, y, cv=kf, scoring='accuracy')
kfold_f1_cv = cross_val_score(cv_model, X, y, cv=kf, scoring='f1_macro')

print("KFold cross_val_score Accuracy:", kfold_acc_cv)
print("KFold cross_val_score F1_macro:", kfold_f1_cv)
print("Mean Accuracy:", kfold_acc_cv.mean())
print("Mean F1_macro:", kfold_f1_cv.mean())

print("\nСовпадение с ручным расчётом:")
print("Accuracy совпадает:", np.allclose(kfold_acc_cv, manual_kfold_acc))
print("F1_macro совпадает:", np.allclose(kfold_f1_cv, manual_kfold_f1))

KFold cross_val_score Accuracy: [1.   0.94 0.94]
KFold cross_val_score F1_macro: [1.         0.94440154 0.93521421]
Mean Accuracy: 0.96
Mean F1_macro: 0.9598719184926082

Совпадение с ручным расчётом:
Accuracy совпадает: True
F1_macro совпадает: True


#### 8. Теперь сделайте k-блочную перекрёстную проверку модели (кросс-валидацию) со стратификацией. Проделайте всё тоже самое, что и в предыдущем пункте.

In [19]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=15)
manual_skf_acc = []
manual_skf_f1 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"Fold {fold}: train={train_idx}, test={test_idx}")
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    fold_model = LogisticRegression(solver='lbfgs', max_iter=1000)
    fold_model.fit(X_train, y_train)
    y_pred = fold_model.predict(X_test)

    manual_skf_acc.append(accuracy_score(y_test, y_pred))
    manual_skf_f1.append(f1_score(y_test, y_pred, average='macro'))

skf_model = LogisticRegression(solver='lbfgs', max_iter=1000)
skf_acc_cv = cross_val_score(skf_model, X, y, cv=skf, scoring='accuracy')
skf_f1_cv = cross_val_score(skf_model, X, y, cv=skf, scoring='f1_macro')

print("\nStratifiedKFold (manual) Accuracy:", manual_skf_acc)
print("StratifiedKFold (manual) F1_macro:", manual_skf_f1)
print("StratifiedKFold (cross_val_score) Accuracy:", skf_acc_cv)
print("StratifiedKFold (cross_val_score) F1_macro:", skf_f1_cv)
print("Mean Accuracy:", skf_acc_cv.mean())
print("Mean F1_macro:", skf_f1_cv.mean())
print("Accuracy совпадает:", np.allclose(skf_acc_cv, manual_skf_acc))
print("F1_macro совпадает:", np.allclose(skf_f1_cv, manual_skf_f1))

Fold 1: train=[  0   1   2   3   4   5   7   8   9  12  14  15  16  17  18  19  21  23
  24  25  26  28  29  31  32  36  37  38  39  40  41  44  45  51  52  53
  54  55  60  61  62  63  65  66  68  70  72  75  77  80  81  82  83  84
  85  86  87  89  90  91  93  94  95  96  97  98 101 102 103 104 105 107
 110 111 112 113 114 116 118 121 122 123 124 126 127 128 129 132 133 134
 139 140 141 142 143 144 145 146 147 149], test=[  6  10  11  13  20  22  27  30  33  34  35  42  43  46  47  48  49  50
  56  57  58  59  64  67  69  71  73  74  76  78  79  88  92  99 100 106
 108 109 115 117 119 120 125 130 131 135 136 137 138 148]
Fold 2: train=[  0   1   2   4   5   6   9  10  11  13  14  15  16  19  20  21  22  23
  26  27  28  29  30  31  33  34  35  42  43  46  47  48  49  50  51  54
  55  56  57  58  59  61  63  64  66  67  69  70  71  73  74  75  76  78
  79  83  84  87  88  89  90  91  92  94  96  97  99 100 101 103 105 106
 107 108 109 110 114 115 116 117 118 119 120 122 125 128 130 13

#### 9. Теперь сделайте перекрёстную проверку, изпользуя leave-one-out разбиение. Проделайте всё тоже самое, что и в предыдущем пункте.

In [23]:
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()
manual_loo_acc = []
manual_loo_f1 = []

for i, (train_idx, test_idx) in enumerate(loo.split(X), start=1):
    if i <= 5:
        print(f"Fold {i}: train_size={len(train_idx)}, test_index={test_idx}")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    fold_model = LogisticRegression(solver='lbfgs', max_iter=1000)
    fold_model.fit(X_train, y_train)
    y_pred = fold_model.predict(X_test)

    manual_loo_acc.append(accuracy_score(y_test, y_pred))
    manual_loo_f1.append(f1_score(y_test, y_pred, average='macro'))

loo_model = LogisticRegression(solver='lbfgs', max_iter=1000)
loo_acc_cv = cross_val_score(loo_model, X, y, cv=loo, scoring='accuracy', n_jobs=-1)
loo_f1_cv = cross_val_score(loo_model, X, y, cv=loo, scoring='f1_macro', n_jobs=-1)

print("\nLOO manual Accuracy (первые 10):", manual_loo_acc[:10])
print("LOO cross_val_score Accuracy (первые 10):", loo_acc_cv[:10])
print("LOO manual F1_macro (первые 10):", manual_loo_f1[:10])
print("LOO cross_val_score F1_macro (первые 10):", loo_f1_cv[:10])

print("\nLOO Mean Accuracy:", np.mean(loo_acc_cv))
print("LOO Mean F1_macro:", np.mean(loo_f1_cv))
print("LOO Std Accuracy:", np.std(loo_acc_cv))
print("LOO Std F1_macro:", np.std(loo_f1_cv))

print("\nСовпадение с ручным расчётом:")
print("Accuracy совпадает:", np.allclose(loo_acc_cv, manual_loo_acc))
print("F1_macro совпадает:", np.allclose(loo_f1_cv, manual_loo_f1))

Fold 1: train_size=149, test_index=[0]
Fold 2: train_size=149, test_index=[1]
Fold 3: train_size=149, test_index=[2]
Fold 4: train_size=149, test_index=[3]
Fold 5: train_size=149, test_index=[4]

LOO manual Accuracy (первые 10): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
LOO cross_val_score Accuracy (первые 10): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
LOO manual F1_macro (первые 10): [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
LOO cross_val_score F1_macro (первые 10): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

LOO Mean Accuracy: 0.9666666666666667
LOO Mean F1_macro: 0.9666666666666667
LOO Std Accuracy: 0.17950549357115014
LOO Std F1_macro: 0.17950549357115014

Совпадение с ручным расчётом:
Accuracy совпадает: True
F1_macro совпадает: True


## Дополнительные задания

#### 1. Изучите разбиение Leave-P-Out. Продемонстрируйте работу этого алгоритма на примере из лабораторной работы.

In [24]:
from sklearn.model_selection import LeavePOut

lpo = LeavePOut(p=2)
lpo_model = LogisticRegression(solver='lbfgs', max_iter=1000)

print("Количество разбиений Leave-P-Out (p=2):", lpo.get_n_splits(X))

for i, (train_idx, test_idx) in enumerate(lpo.split(X), start=1):
    if i <= 5:
        print(f"Split {i}: train_size={len(train_idx)}, test_indices={test_idx}")
    else:
        break

lpo_acc = cross_val_score(lpo_model, X, y, cv=lpo, scoring='accuracy', n_jobs=-1)
print("\nLeave-P-Out Accuracy (первые 10):", lpo_acc[:10])
print("Leave-P-Out Mean Accuracy:", lpo_acc.mean())
print("Leave-P-Out Std Accuracy:", lpo_acc.std())

Количество разбиений Leave-P-Out (p=2): 11175
Split 1: train_size=148, test_indices=[0 1]
Split 2: train_size=148, test_indices=[0 2]
Split 3: train_size=148, test_indices=[0 3]
Split 4: train_size=148, test_indices=[0 4]
Split 5: train_size=148, test_indices=[0 5]

Leave-P-Out Accuracy (первые 10): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Leave-P-Out Mean Accuracy: 0.9654586129753915
Leave-P-Out Std Accuracy: 0.12872356473113936


#### 2. Изучите функцию cross_validate(). Продемонстрируйте работу этой функции на тех же данных.

In [25]:
from sklearn.model_selection import cross_validate

cv_scores = cross_validate(
    LogisticRegression(solver='lbfgs', max_iter=1000),
    X,
    y,
    cv=kf,
    scoring=['accuracy', 'f1_macro'],
    return_train_score=True,
    n_jobs=-1
)

print("Ключи результата cross_validate:", sorted(cv_scores.keys()))
print("Test Accuracy:", cv_scores['test_accuracy'])
print("Train Accuracy:", cv_scores['train_accuracy'])
print("Test F1_macro:", cv_scores['test_f1_macro'])
print("Train F1_macro:", cv_scores['train_f1_macro'])
print("Mean test Accuracy:", cv_scores['test_accuracy'].mean())
print("Mean test F1_macro:", cv_scores['test_f1_macro'].mean())
print("Mean fit_time:", cv_scores['fit_time'].mean())

Ключи результата cross_validate: ['fit_time', 'score_time', 'test_accuracy', 'test_f1_macro', 'train_accuracy', 'train_f1_macro']
Test Accuracy: [1.   0.94 0.94]
Train Accuracy: [0.98 0.99 0.97]
Test F1_macro: [1.         0.94440154 0.93521421]
Train F1_macro: [0.98005952 0.98956039 0.97099012]
Mean test Accuracy: 0.96
Mean test F1_macro: 0.9598719184926082
Mean fit_time: 0.010923147201538086


#### 3. Оцените при помощи кросс-валидации другие метрики эффективности для этой же модели.

In [26]:
extra_metrics = ['accuracy', 'balanced_accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

extra_cv = cross_validate(
    LogisticRegression(solver='lbfgs', max_iter=1000),
    X,
    y,
    cv=skf,
    scoring=extra_metrics,
    n_jobs=-1
)

print("Дополнительные метрики (среднее по StratifiedKFold):")
for metric in extra_metrics:
    values = extra_cv[f'test_{metric}']
    print(f"{metric}: mean={values.mean():.4f}, std={values.std():.4f}")

Дополнительные метрики (среднее по StratifiedKFold):
accuracy: mean=0.9600, std=0.0000
balanced_accuracy: mean=0.9596, std=0.0010
precision_macro: mean=0.9631, std=0.0025
recall_macro: mean=0.9596, std=0.0010
f1_macro: mean=0.9598, std=0.0006


#### 4. Сравните кросс-валидированные результаты работы нескольких моделей на одних и тех же данных.

In [27]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

models = {
    'LogisticRegression': LogisticRegression(solver='lbfgs', max_iter=1000),
    'SVC_rbf': SVC(kernel='rbf', gamma='scale'),
    'KNN_k5': KNeighborsClassifier(n_neighbors=5),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
}

model_results = []
for model_name, model_obj in models.items():
    model_scores = cross_val_score(model_obj, X, y, cv=skf, scoring='f1_macro', n_jobs=-1)
    model_results.append((model_name, model_scores.mean(), model_scores.std()))
    print(f"{model_name}: mean_f1_macro={model_scores.mean():.4f}, std={model_scores.std():.4f}")

comparison_df = pd.DataFrame(model_results, columns=['model', 'mean_f1_macro', 'std_f1_macro'])
comparison_df = comparison_df.sort_values('mean_f1_macro', ascending=False).reset_index(drop=True)

print("\nСравнение моделей:")
display(comparison_df)

LogisticRegression: mean_f1_macro=0.9598, std=0.0006
SVC_rbf: mean_f1_macro=0.9598, std=0.0166
KNN_k5: mean_f1_macro=0.9464, std=0.0091
DecisionTree: mean_f1_macro=0.9598, std=0.0006

Сравнение моделей:


,model,mean_f1_macro,std_f1_macro
0,SVC_rbf,0.959847,0.016565
1,LogisticRegression,0.959822,0.000597
2,DecisionTree,0.959822,0.000597
3,KNN_k5,0.946382,0.009128


#### 5. Повторите анализ на другом датасете: встроенном наборе данных о диабете.

In [28]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

diabetes = load_diabetes()
X_diab, y_diab = diabetes.data, diabetes.target

diab_kf = KFold(n_splits=5, shuffle=True, random_state=42)
reg_models = {
    'LinearRegression': LinearRegression(),
    'Ridge(alpha=1.0)': Ridge(alpha=1.0),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=300, random_state=42),
}

diab_results = []
for model_name, model_obj in reg_models.items():
    scores = cross_validate(
        model_obj,
        X_diab,
        y_diab,
        cv=diab_kf,
        scoring=('r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'),
        n_jobs=-1
    )
    mean_r2 = scores['test_r2'].mean()
    mean_mae = -scores['test_neg_mean_absolute_error'].mean()
    mean_rmse = -scores['test_neg_root_mean_squared_error'].mean()
    diab_results.append((model_name, mean_r2, mean_mae, mean_rmse))
    print(f"{model_name}: R2={mean_r2:.4f}, MAE={mean_mae:.4f}, RMSE={mean_rmse:.4f}")

diab_df = pd.DataFrame(diab_results, columns=['model', 'mean_r2', 'mean_mae', 'mean_rmse'])
diab_df = diab_df.sort_values('mean_r2', ascending=False).reset_index(drop=True)

print("\nРезультаты на датасете diabetes:")
display(diab_df)

LinearRegression: R2=0.4785, MAE=44.2697, RMSE=54.8489
Ridge(alpha=1.0): R2=0.4093, MAE=49.0054, RMSE=58.5642
RandomForestRegressor: R2=0.4272, MAE=46.7665, RMSE=57.5526

Результаты на датасете diabetes:


,model,mean_r2,mean_mae,mean_rmse
0,LinearRegression,0.478470,44.269746,54.848941
1,RandomForestRegressor,0.427227,46.766546,57.552557
2,Ridge(alpha=1.0),0.409300,49.005365,58.564192


### 6. Сделайте k-блочную перекрёстную проверку (KFold) модели логистической регрессии, предварительно стандартизировав данные. Для этого создайте конвейер с помощью make_pipeline из библиотеки sklearn.pipeline, который будет стандартизировать, а затем выполнять логистическую регрессию.

In [29]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver='lbfgs', max_iter=1000)
 )

pipeline_scores = cross_val_score(
    pipeline_model,
    X,
    y,
    cv=kf,
    scoring='f1_macro',
    n_jobs=-1
)

base_scores = cross_val_score(
    LogisticRegression(solver='lbfgs', max_iter=1000),
    X,
    y,
    cv=kf,
    scoring='f1_macro',
    n_jobs=-1
)

print("Pipeline (StandardScaler + LogisticRegression) F1_macro:", pipeline_scores)
print("Pipeline mean F1_macro:", pipeline_scores.mean())
print("Base LogisticRegression F1_macro:", base_scores)
print("Base mean F1_macro:", base_scores.mean())
print("Δ(mean F1_macro):", pipeline_scores.mean() - base_scores.mean())

Pipeline (StandardScaler + LogisticRegression) F1_macro: [1.         0.94440154 0.89556088]
Pipeline mean F1_macro: 0.9466541426218845
Base LogisticRegression F1_macro: [1.         0.94440154 0.93521421]
Base mean F1_macro: 0.9598719184926082
Δ(mean F1_macro): -0.013217775870723703
